# wf2020 recovery participation target v1

- **Baselines**: canonical `v19reg_v22_budget_usage_bull_guard_v1`; direct comparisons `cap10_three_fold_checkpoint_metric_replay_v2`, `wf2020_checkpoint_near_miss_overlay_screen_v1`, and `fold_local_bull_reentry_participation_target_v1`.
- **Role**: `diagnose` and target handoff. This notebook creates a non-leaking fold-local target contract for the next checkpoint replay; it does not retain or promote an actor.
- **Target window**: `wf_2020` failure repair, with three-fold target coverage kept for `wf_2020_train`, `wf_2022_train`, and `wf_2023_2024_train`.
- **Changed layer**: `model_selection`, `training_target_composition`, and `bc_checkpoint_target_contract`.
- **External reference**: `soxlbacktest_qlib_buysell_reference` remains the buy/sell baseline; the new target is a recovery-participation overlay on the existing bull-reentry contract.
- **Hypothesis**: the wf2020 near-miss path failed because pure drawdown caps removed recovery participation. A target that caps risk-only drawdown days but preserves recovery days should give the next replay a better objective than either low-return BC initialization or unconstrained ppo retained iter 4.
- **Acceptance criteria**: write a three-fold target contract, write a wf2020 surrogate preview, expose BUY/SELL feature-state comparisons, keep `promotion_allowed=false`, and point to the next replay mechanism.
- **Historical success/failure check**: closest success is `cap10_three_fold_checkpoint_metric_replay_v2` passing `wf_2022` and `wf_2023_2024`; closest failure is `wf2020_checkpoint_near_miss_overlay_screen_v1`, where no post-hoc overlay passed both return and MaxDD. This run is not a duplicate because it produces a training target contract rather than another post-hoc dashboard overlay.

In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "SPEC.md").exists() and (candidate / "runtime").exists():
            return candidate
    raise RuntimeError(f"repo root not found from {start}")

ROOT = find_repo_root(Path.cwd().resolve())
EXPERIMENT = "wf2020_recovery_participation_target_v1"
CARD = ROOT / "runtime/model_selection/card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

NEAR_MISS_DAILY_PATH = ROOT / "runtime/model_selection/card/cap10_three_fold_checkpoint_metric_replay_v2/official_daily_paths/wf_2020__ppo_retained_iter_4_official_daily_path.csv"
BC_SAFE_DAILY_PATH = ROOT / "runtime/model_selection/card/cap10_three_fold_checkpoint_metric_replay_v2/official_daily_paths/wf_2020__bc_initialization_official_daily_path.csv"
BULL_REENTRY_CONTRACT_PATH = ROOT / "runtime/model_selection/card/fold_local_bull_reentry_participation_target_v1/bull_reentry_three_fold_target_contract.csv"
OVERLAY_SUMMARY_PATH = ROOT / "runtime/model_selection/card/wf2020_checkpoint_near_miss_overlay_screen_v1/summary.json"
FRONTIER_SUMMARY_PATH = ROOT / "runtime/model_selection/card/fold_frontier_failure_attribution_v1/summary.json"

TARGET_RETURN_PCT = 69.60455156850816
TARGET_MAX_DD_PCT = 20.0

SELECTED_RECIPE = {
    "strategy_dd_threshold": 0.12,
    "risk_cap": 0.15,
    "recovery_floor": 0.65,
    "conflict_bridge_floor": 0.35,
    "recovery_mom5_threshold": 0.02,
    "scale_clip": 3.0,
}

SIDE_CARS = {
    "surrogate_grid": CARD / "surrogate_grid.csv",
    "selected_preview_daily": CARD / "selected_recovery_preview_daily.csv",
    "target_contract": CARD / "recovery_participation_three_fold_target_contract.csv",
    "target_audit": CARD / "target_audit.csv",
    "buy_sell_feature_comparison": CARD / "buy_sell_feature_comparison.csv",
    "state_action_comparison": CARD / "state_action_comparison.csv",
    "selection": CARD / "selection.json",
    "summary": CARD / "summary.json",
    "conclusion": CARD / "conclusion.md",
}

def rel(path: Path) -> str:
    return str(path.relative_to(ROOT))

def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, sort_keys=True) + "\n", encoding="utf-8")

print(f"card={rel(CARD)}")

card=runtime/model_selection/card/wf2020_recovery_participation_target_v1


In [2]:
near = pd.read_csv(NEAR_MISS_DAILY_PATH, parse_dates=["datetime"])
bc_safe = pd.read_csv(BC_SAFE_DAILY_PATH, parse_dates=["datetime"])
source_contract = pd.read_csv(BULL_REENTRY_CONTRACT_PATH, parse_dates=["datetime", "train_start", "train_end", "eval_start", "eval_end"])
frontier_summary = json.loads(FRONTIER_SUMMARY_PATH.read_text(encoding="utf-8"))
overlay_summary = json.loads(OVERLAY_SUMMARY_PATH.read_text(encoding="utf-8"))

required_near_cols = {"datetime", "strategy_nav", "price", "buyhold_nav", "position", "action"}
required_contract_cols = {"datetime", "fold", "train_start", "train_end", "eval_start", "eval_end", "teacher_position", "risk_target_position", "strategy_drawdown", "price"}
missing_near = required_near_cols - set(near.columns)
missing_contract = required_contract_cols - set(source_contract.columns)
assert not missing_near, missing_near
assert not missing_contract, missing_contract

print("near_miss", near.datetime.min().date(), near.datetime.max().date(), near.shape)
print("source_contract", source_contract.datetime.min().date(), source_contract.datetime.max().date(), source_contract.shape)
print("folds", sorted(source_contract.fold.astype(str).unique()))
print("frontier_return_floor", frontier_summary.get("wf2020_reference_floor_return_pct"), "target_max_dd", frontier_summary.get("target_max_dd_pct"))

near_miss 2020-01-02 2020-12-30 (252, 8)
source_contract 2010-03-11 2022-12-30 (3731, 39)
folds ['wf_2020_train', 'wf_2022_train', 'wf_2023_2024_train']
frontier_return_floor 69.60455156850816 target_max_dd 20.0


In [3]:
def max_drawdown_pct(nav: pd.Series) -> float:
    nav = nav.astype(float)
    peak = nav.cummax()
    dd = (peak - nav) / peak
    return float(dd.max() * 100.0)

def return_pct(nav: pd.Series) -> float:
    nav = nav.astype(float)
    if nav.empty or float(nav.iloc[0]) == 0:
        return float("nan")
    return float((nav.iloc[-1] / nav.iloc[0] - 1.0) * 100.0)

def enrich_preview_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values("datetime").reset_index(drop=True).copy()
    out["base_ret"] = out["strategy_nav"].pct_change().fillna(0.0)
    prior_nav = out["strategy_nav"].shift(1)
    out["strategy_dd_prev"] = (prior_nav / prior_nav.cummax() - 1.0).fillna(0.0)
    out["soxl_mom5_prev"] = out["price"].pct_change(5).shift(1).fillna(0.0)
    prior_price = out["price"].shift(1)
    out["soxl_dd20_prev"] = (prior_price / prior_price.rolling(20, min_periods=1).max() - 1.0).fillna(0.0)
    out["position_prev"] = out["position"].shift(1).fillna(out["position"])
    return out

def apply_recipe_to_preview(df: pd.DataFrame, recipe: dict) -> pd.DataFrame:
    out = enrich_preview_features(df)
    risk = out["strategy_dd_prev"] <= -float(recipe["strategy_dd_threshold"])
    recovery = (out["strategy_dd_prev"] <= -0.05) & (out["soxl_mom5_prev"] >= float(recipe["recovery_mom5_threshold"]))
    conflict = risk & recovery
    risk_only = risk & ~recovery
    recovery_only = recovery & ~risk

    target = out["position"].astype(float).copy()
    target.loc[risk_only] = np.minimum(target.loc[risk_only], float(recipe["risk_cap"]))
    target.loc[conflict] = np.minimum(
        np.maximum(target.loc[conflict], float(recipe["conflict_bridge_floor"])),
        max(float(recipe["conflict_bridge_floor"]), float(recipe["recovery_floor"])),
    )
    target.loc[recovery_only] = np.maximum(target.loc[recovery_only], float(recipe["recovery_floor"]))

    denom = out["position"].astype(float).replace(0.0, np.nan)
    scale = (target / denom).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    scale = scale.clip(lower=0.0, upper=float(recipe["scale_clip"]))
    out["target_position"] = target.clip(lower=0.0, upper=1.0)
    out["target_delta"] = out["target_position"] - out["position"].astype(float)
    out["risk_control_state"] = risk.astype(bool)
    out["recovery_participation_state"] = recovery.astype(bool)
    out["drawdown_recovery_conflict_state"] = conflict.astype(bool)
    out["target_scale"] = scale
    out["target_ret"] = out["base_ret"] * out["target_scale"]
    out["target_nav"] = (1.0 + out["target_ret"]).cumprod() * float(out["strategy_nav"].iloc[0])
    out["target_drawdown"] = out["target_nav"] / out["target_nav"].cummax() - 1.0
    return out

def preview_metrics(df: pd.DataFrame) -> dict[str, float | int | bool]:
    return {
        "return_pct": return_pct(df["target_nav"]),
        "max_dd_pct": max_drawdown_pct(df["target_nav"]),
        "risk_days": int(df["risk_control_state"].sum()),
        "recovery_days": int(df["recovery_participation_state"].sum()),
        "conflict_days": int(df["drawdown_recovery_conflict_state"].sum()),
        "avg_scale": float(df["target_scale"].mean()),
        "min_scale": float(df["target_scale"].min()),
        "max_scale": float(df["target_scale"].max()),
    }

base_near_metrics = {
    "return_pct": return_pct(near["strategy_nav"]),
    "max_dd_pct": max_drawdown_pct(near["strategy_nav"]),
}
bc_safe_metrics = {
    "return_pct": return_pct(bc_safe["strategy_nav"]),
    "max_dd_pct": max_drawdown_pct(bc_safe["strategy_nav"]),
}
print("near_miss", base_near_metrics)
print("bc_safe", bc_safe_metrics)

near_miss {'return_pct': 61.32404958994388, 'max_dd_pct': 20.918285087095683}
bc_safe {'return_pct': 40.58032598940371, 'max_dd_pct': 14.984700509769244}


In [4]:
grid_rows = []
for dd_threshold in [0.08, 0.10, 0.12, 0.14, 0.16]:
    for risk_cap in [0.15, 0.20, 0.25, 0.30, 0.35]:
        for recovery_floor in [0.45, 0.55, 0.65, 0.75, 0.85]:
            for bridge in [0.35, 0.45, 0.55, 0.65, 0.75]:
                for mom_threshold in [-0.02, 0.00, 0.02, 0.04]:
                    recipe = {
                        "strategy_dd_threshold": dd_threshold,
                        "risk_cap": risk_cap,
                        "recovery_floor": recovery_floor,
                        "conflict_bridge_floor": bridge,
                        "recovery_mom5_threshold": mom_threshold,
                        "scale_clip": SELECTED_RECIPE["scale_clip"],
                    }
                    preview = apply_recipe_to_preview(near, recipe)
                    metrics = preview_metrics(preview)
                    row = {**recipe, **metrics}
                    row["return_floor_gap_pp"] = TARGET_RETURN_PCT - row["return_pct"]
                    row["max_dd_gap_pct"] = row["max_dd_pct"] - TARGET_MAX_DD_PCT
                    row["return_floor_gate"] = bool(row["return_pct"] >= TARGET_RETURN_PCT)
                    row["max_dd_gate"] = bool(row["max_dd_pct"] <= TARGET_MAX_DD_PCT)
                    row["surrogate_gate"] = bool(row["return_floor_gate"] and row["max_dd_gate"])
                    grid_rows.append(row)

grid = pd.DataFrame(grid_rows)
grid["selected_recipe"] = False
selected_mask = np.ones(len(grid), dtype=bool)
for key, value in SELECTED_RECIPE.items():
    if key == "scale_clip":
        continue
    selected_mask &= np.isclose(grid[key].astype(float), float(value))
assert selected_mask.sum() == 1, selected_mask.sum()
grid.loc[selected_mask, "selected_recipe"] = True
selected_grid_row = grid.loc[selected_mask].iloc[0].to_dict()

grid_sorted = grid.sort_values(
    ["surrogate_gate", "selected_recipe", "max_dd_gate", "return_floor_gate", "return_pct"],
    ascending=[False, False, False, False, False],
)
grid_sorted.to_csv(SIDE_CARS["surrogate_grid"], index=False)

print("grid_rows", len(grid), "surrogate_pass", int(grid["surrogate_gate"].sum()))
print(pd.DataFrame([selected_grid_row]).to_string(index=False))
display(grid_sorted.head(12))

grid_rows 2500 surrogate_pass 551
 strategy_dd_threshold  risk_cap  recovery_floor  conflict_bridge_floor  recovery_mom5_threshold  scale_clip  return_pct  max_dd_pct  risk_days  recovery_days  conflict_days  avg_scale  min_scale  max_scale  return_floor_gap_pp  max_dd_gap_pct  return_floor_gate  max_dd_gate  surrogate_gate  selected_recipe
                  0.12      0.15            0.65                   0.35                     0.02         3.0   73.074892   15.354491         16             16              5   0.988192   0.225773   1.459433             -3.47034       -4.645509               True         True            True             True


,strategy_dd_threshold,risk_cap,recovery_floor,conflict_bridge_floor,recovery_mom5_threshold,scale_clip,return_pct,max_dd_pct,risk_days,recovery_days,conflict_days,avg_scale,min_scale,max_scale,return_floor_gap_pp,max_dd_gap_pct,return_floor_gate,max_dd_gate,surrogate_gate,selected_recipe
1042,0.12,0.15,0.65,0.35,0.02,3.0,73.074892,15.354491,16,16,5,0.988192,0.225773,1.459433,-3.470340,-4.645509,True,True,True,True
1492,0.12,0.35,0.85,0.65,-0.02,3.0,94.372256,19.923983,16,20,6,1.042887,0.526803,1.908489,-24.767705,-0.076017,True,True,True,False
1392,0.12,0.30,0.85,0.65,-0.02,3.0,94.279302,19.158959,16,20,6,1.039124,0.451545,1.908489,-24.674751,-0.841041,True,True,True,False
1396,0.12,0.30,0.85,0.75,-0.02,3.0,94.210626,19.452504,16,20,6,1.043771,0.451545,1.908489,-24.606074,-0.547496,True,True,True,False
1292,0.12,0.25,0.85,0.65,-0.02,3.0,94.155994,18.393935,16,20,6,1.035362,0.376288,1.908489,-24.551443,-1.606065,True,True,True,False
1296,0.12,0.25,0.85,0.75,-0.02,3.0,94.087361,18.690258,16,20,6,1.040009,0.376288,1.908489,-24.482810,-1.309742,True,True,True,False
1488,0.12,0.35,0.85,0.55,-0.02,3.0,94.007724,19.633215,16,20,6,1.039367,0.526803,1.908489,-24.403173,-0.366785,True,True,True,False
1192,0.12,0.20,0.85,0.65,-0.02,3.0,94.002634,17.628910,16,20,6,1.031599,0.301030,1.908489,-24.398083,-2.371090,True,True,True,False
1196,0.12,0.20,0.85,0.75,-0.02,3.0,93.934056,17.977985,16,20,6,1.036246,0.301030,1.908489,-24.329504,-2.022015,True,True,True,False
1388,0.12,0.30,0.85,0.55,-0.02,3.0,93.914944,18.865413,16,20,6,1.035604,0.451545,1.908489,-24.310393,-1.134587,True,True,True,False


In [5]:
selected_preview = apply_recipe_to_preview(near, SELECTED_RECIPE)
selected_preview_metrics = preview_metrics(selected_preview)
selected_preview_metrics.update({
    "return_floor_gap_pp": TARGET_RETURN_PCT - selected_preview_metrics["return_pct"],
    "max_dd_gap_pct": selected_preview_metrics["max_dd_pct"] - TARGET_MAX_DD_PCT,
    "return_floor_gate": bool(selected_preview_metrics["return_pct"] >= TARGET_RETURN_PCT),
    "max_dd_gate": bool(selected_preview_metrics["max_dd_pct"] <= TARGET_MAX_DD_PCT),
})
selected_preview_metrics["surrogate_gate"] = bool(selected_preview_metrics["return_floor_gate"] and selected_preview_metrics["max_dd_gate"])

preview_cols = [
    "datetime", "strategy_nav", "target_nav", "buyhold_nav", "price", "action", "position", "target_position", "target_delta",
    "strategy_dd_prev", "soxl_mom5_prev", "soxl_dd20_prev", "risk_control_state", "recovery_participation_state",
    "drawdown_recovery_conflict_state", "target_scale", "base_ret", "target_ret", "target_drawdown",
]
selected_preview[preview_cols].to_csv(SIDE_CARS["selected_preview_daily"], index=False)

feature_cols = ["position", "target_position", "target_delta", "strategy_dd_prev", "soxl_mom5_prev", "soxl_dd20_prev", "target_scale"]
comparison = (
    selected_preview.assign(action=selected_preview["action"].fillna("HOLD").astype(str).str.upper())
    .groupby("action", dropna=False)
    .agg(
        rows=("datetime", "count"),
        risk_days=("risk_control_state", "sum"),
        recovery_days=("recovery_participation_state", "sum"),
        conflict_days=("drawdown_recovery_conflict_state", "sum"),
        **{f"mean_{col}": (col, "mean") for col in feature_cols},
        **{f"median_{col}": (col, "median") for col in feature_cols},
    )
    .reset_index()
)
comparison.to_csv(SIDE_CARS["buy_sell_feature_comparison"], index=False)

state_action = (
    selected_preview.assign(action=selected_preview["action"].fillna("HOLD").astype(str).str.upper())
    .groupby(["action", "risk_control_state", "recovery_participation_state", "drawdown_recovery_conflict_state"], dropna=False)
    .agg(rows=("datetime", "count"), mean_position=("position", "mean"), mean_target_position=("target_position", "mean"), mean_target_delta=("target_delta", "mean"))
    .reset_index()
)
state_action.to_csv(SIDE_CARS["state_action_comparison"], index=False)

print(selected_preview_metrics)
display(comparison)
display(state_action.sort_values(["action", "rows"], ascending=[True, False]).head(20))

{'return_pct': 73.07489183483915, 'max_dd_pct': 15.354491077731963, 'risk_days': 16, 'recovery_days': 16, 'conflict_days': 5, 'avg_scale': 0.9881924753483478, 'min_scale': 0.2257725883269507, 'max_scale': 1.4594328102461482, 'return_floor_gap_pp': -3.470340266330993, 'max_dd_gap_pct': -4.6455089222680375, 'return_floor_gate': True, 'max_dd_gate': True, 'surrogate_gate': True}


,action,rows,risk_days,recovery_days,conflict_days,mean_position,mean_target_position,mean_target_delta,mean_strategy_dd_prev,mean_soxl_mom5_prev,mean_soxl_dd20_prev,mean_target_scale,median_position,median_target_position,median_target_delta,median_strategy_dd_prev,median_soxl_mom5_prev,median_soxl_dd20_prev,median_target_scale
0,BUY,12,1,1,0,0.453223,0.427039,-0.026183,-0.041200,0.076599,-0.127446,0.972283,0.451006,0.451006,0.0,-0.019865,0.060814,-0.096281,1.0
1,HOLD,221,14,14,5,0.352712,0.345429,-0.007283,-0.026492,0.013576,-0.131674,0.989173,0.447429,0.289974,0.0,-0.004565,0.024979,-0.060709,1.0
2,SELL,19,1,1,0,0.394661,0.387752,-0.006910,-0.029361,0.099195,-0.061511,0.986840,0.451006,0.451006,0.0,-0.004754,0.101966,-0.024781,1.0


,action,risk_control_state,recovery_participation_state,drawdown_recovery_conflict_state,rows,mean_position,mean_target_position,mean_target_delta
0,BUY,False,False,False,10,0.432447,0.432447,0.000000
1,BUY,False,True,False,1,0.451006,0.650000,0.198994
2,BUY,True,False,False,1,0.663193,0.150000,-0.513193
3,HOLD,False,False,False,198,0.335789,0.335789,0.000000
4,HOLD,False,True,False,9,0.450381,0.650000,0.199619
5,HOLD,True,False,False,9,0.526948,0.150000,-0.376948
6,HOLD,True,True,True,5,0.533449,0.530711,-0.002738
7,SELL,False,False,False,17,0.386311,0.386311,0.000000
8,SELL,False,True,False,1,0.452419,0.650000,0.197581
9,SELL,True,False,False,1,0.478863,0.150000,-0.328863


In [6]:
def _bool_series(frame: pd.DataFrame, column: str) -> pd.Series:
    if column not in frame:
        return pd.Series(False, index=frame.index)
    return frame[column].fillna(False).astype(bool)

def build_contract_targets(source: pd.DataFrame, recipe: dict) -> pd.DataFrame:
    out = source.sort_values(["fold", "datetime"]).reset_index(drop=True).copy()
    out["source_risk_target_position"] = out["risk_target_position"].astype(float)
    out["source_rebound_target_position"] = out.get("rebound_target_position", out["risk_target_position"]).astype(float)

    parts = []
    for fold, group in out.groupby("fold", sort=False):
        g = group.copy()
        g["strategy_drawdown_prev"] = g["strategy_drawdown"].astype(float).shift(1).fillna(0.0)
        g["soxl_mom5_prev"] = g["price"].astype(float).pct_change(5).shift(1).fillna(0.0)
        prior_price = g["price"].astype(float).shift(1)
        g["soxl_dd20_prev"] = (prior_price / prior_price.rolling(20, min_periods=1).max() - 1.0).fillna(0.0)

        risk = g["strategy_drawdown_prev"] >= float(recipe["strategy_dd_threshold"])
        recovery = (g["strategy_drawdown_prev"] >= 0.05) & (g["soxl_mom5_prev"] >= float(recipe["recovery_mom5_threshold"]))
        downtrend = _bool_series(g, "is_downtrend_state")
        recovery = recovery & ~downtrend
        conflict = risk & recovery
        risk_only = risk & ~recovery
        recovery_only = recovery & ~risk

        target = g["source_risk_target_position"].astype(float).copy()
        target.loc[risk_only] = np.minimum(target.loc[risk_only], float(recipe["risk_cap"]))
        target.loc[conflict] = np.minimum(
            np.maximum(target.loc[conflict], float(recipe["conflict_bridge_floor"])),
            max(float(recipe["conflict_bridge_floor"]), float(recipe["recovery_floor"])),
        )
        target.loc[recovery_only] = np.maximum(target.loc[recovery_only], float(recipe["recovery_floor"]))
        target = target.clip(lower=0.0, upper=1.0)

        g["risk_control_state_v2"] = risk.astype(bool)
        g["recovery_participation_state_v2"] = recovery.astype(bool)
        g["drawdown_recovery_conflict_state_v2"] = conflict.astype(bool)
        g["risk_only_state_v2"] = risk_only.astype(bool)
        g["recovery_only_state_v2"] = recovery_only.astype(bool)
        g["recovery_participation_target_position"] = target
        g["risk_target_position"] = target
        g["rebound_target_position"] = np.maximum(g["rebound_target_position"].astype(float), target).clip(lower=0.0, upper=1.0)
        g["target_delta_vs_source"] = g["risk_target_position"] - g["source_risk_target_position"]
        g["target_semantics_revision"] = "wf2020_recovery_participation_drawdown_state_v1"
        g["source_contract_id"] = "fold_local_bull_reentry_participation_target_v1"
        g["target_recipe_id"] = "dd12_cap15_recovery65_bridge35_mom02"
        g["source_components"] = g.get("source_components", "").astype(str) + ",recovery_participation_drawdown_state_v1"
        g["source_component_count"] = g.get("source_component_count", 0).fillna(0).astype(int) + 1
        g["objective_weight"] = np.where(
            g["drawdown_recovery_conflict_state_v2"],
            np.maximum(g["objective_weight"].astype(float), 8.0),
            np.where(
                g["risk_control_state_v2"] | g["recovery_participation_state_v2"],
                np.maximum(g["objective_weight"].astype(float), 6.0),
                g["objective_weight"].astype(float),
            ),
        )
        g["target_reason"] = np.select(
            [g["drawdown_recovery_conflict_state_v2"], g["risk_only_state_v2"], g["recovery_only_state_v2"]],
            ["drawdown_recovery_bridge", "risk_only_cap", "recovery_participation_floor"],
            default="source_contract",
        )
        parts.append(g)
    return pd.concat(parts, ignore_index=True)

contract = build_contract_targets(source_contract, SELECTED_RECIPE)
contract["eval_leakage_row"] = (contract["datetime"] >= contract["eval_start"]) & (contract["datetime"] <= contract["eval_end"])
contract.to_csv(SIDE_CARS["target_contract"], index=False)

audit = (
    contract.groupby("fold", dropna=False)
    .agg(
        rows=("datetime", "count"),
        start=("datetime", "min"),
        end=("datetime", "max"),
        train_start=("train_start", "min"),
        train_end=("train_end", "max"),
        eval_start=("eval_start", "min"),
        eval_end=("eval_end", "max"),
        modified_rows=("target_delta_vs_source", lambda s: int((s.abs() > 1e-9).sum())),
        positive_lift_rows=("target_delta_vs_source", lambda s: int((s > 1e-9).sum())),
        negative_cap_rows=("target_delta_vs_source", lambda s: int((s < -1e-9).sum())),
        risk_rows=("risk_control_state_v2", "sum"),
        recovery_rows=("recovery_participation_state_v2", "sum"),
        conflict_rows=("drawdown_recovery_conflict_state_v2", "sum"),
        mean_source_target=("source_risk_target_position", "mean"),
        mean_target=("risk_target_position", "mean"),
        p90_target=("risk_target_position", lambda s: float(s.quantile(0.9))),
        max_target=("risk_target_position", "max"),
        eval_leakage_rows=("eval_leakage_row", "sum"),
    )
    .reset_index()
)
for col in ["start", "end", "train_start", "train_end", "eval_start", "eval_end"]:
    audit[col] = pd.to_datetime(audit[col]).dt.strftime("%Y-%m-%d")
audit.to_csv(SIDE_CARS["target_audit"], index=False)

print("target_rows", len(contract))
display(audit)

target_rows 3731


/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_96175/2037144293.py:15: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  g["soxl_mom5_prev"] = g["price"].astype(float).pct_change(5).shift(1).fillna(0.0)
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_96175/2037144293.py:15: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  g["soxl_mom5_prev"] = g["price"].astype(float).pct_change(5).shift(1).fillna(0.0)


,fold,rows,start,end,train_start,train_end,eval_start,eval_end,modified_rows,positive_lift_rows,negative_cap_rows,risk_rows,recovery_rows,conflict_rows,mean_source_target,mean_target,p90_target,max_target,eval_leakage_rows
0,wf_2020_train,2470,2010-03-11,2019-12-31,2010-03-11,2019-12-31,2020-01-01,2020-12-31,949,156,793,839,226,54,0.330183,0.304600,0.650000,0.915100,0
1,wf_2022_train,505,2020-01-02,2021-12-31,2020-01-02,2021-12-31,2022-01-03,2022-12-30,0,0,0,0,0,0,0.348817,0.348817,0.742263,0.893943,0
2,wf_2023_2024_train,756,2020-01-02,2022-12-30,2020-01-02,2022-12-30,2023-01-03,2024-12-31,0,0,0,0,0,0,0.277530,0.277530,0.636353,0.893943,0


In [7]:
sidecars = [rel(path) for path in SIDE_CARS.values()]
selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": "wf2020_recovery_participation_target_ready_for_checkpoint_replay_no_promotion",
    "selected_recipe": SELECTED_RECIPE,
    "selected_recipe_id": "dd12_cap15_recovery65_bridge35_mom02",
    "direct_comparison_baseline": [
        "cap10_three_fold_checkpoint_metric_replay_v2:ppo_retained_iter_4",
        "wf2020_checkpoint_near_miss_overlay_screen_v1",
        "fold_local_bull_reentry_participation_target_v1",
    ],
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "reference_baseline": "soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05",
    "closest_historical_success_case": "cap10_three_fold_checkpoint_metric_replay_v2 passed wf_2022 and wf_2023_2024 but failed wf_2020.",
    "closest_historical_failure_case": "wf2020_checkpoint_near_miss_overlay_screen_v1 found no post-hoc overlay passing both wf2020 return and MaxDD gates.",
    "why_not_duplicate": "This notebook writes a non-leaking three-fold target contract and BUY/SELL feature-state sidecars, not another post-hoc overlay screen.",
    "role": "diagnose",
    "changed_layer": ["model_selection", "training_target_composition", "bc_checkpoint_target_contract"],
    "target_window": "wf_2020",
    "target_return_pct": TARGET_RETURN_PCT,
    "target_max_dd_pct": TARGET_MAX_DD_PCT,
    "base_near_miss_metrics": base_near_metrics,
    "bc_safe_metrics": bc_safe_metrics,
    "selected_preview_metrics": selected_preview_metrics,
    "target_contract_path": rel(SIDE_CARS["target_contract"]),
    "target_audit_path": rel(SIDE_CARS["target_audit"]),
    "buy_sell_feature_comparison_path": rel(SIDE_CARS["buy_sell_feature_comparison"]),
    "surrogate_grid_path": rel(SIDE_CARS["surrogate_grid"]),
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "training_allowed_from_this_card": False,
    "checkpoint_replay_allowed_next": True,
    "leakage_guard": {
        "target_contract": "uses only fold-local train rows from the existing bull-reentry contract; no 2020 eval rows are written as training labels",
        "preview_daily": "uses 2020 near-miss official path only as surrogate diagnostics and dashboard/readout evidence",
        "eval_leakage_rows": int(audit["eval_leakage_rows"].sum()),
    },
    "next_mechanism": "rerun_three_fold_checkpoint_replay_with_recovery_participation_target_contract",
    "sidecars": [Path(item).name for item in sidecars],
}

summary = dict(selection)
summary.update({
    "source_overlay_decision": overlay_summary.get("decision"),
    "source_frontier_decision": frontier_summary.get("decision"),
    "surrogate_pass_count": int(grid["surrogate_gate"].sum()),
    "surrogate_variant_count": int(len(grid)),
    "target_contract_rows": int(len(contract)),
    "target_modified_rows": int((contract["target_delta_vs_source"].abs() > 1e-9).sum()),
    "target_eval_leakage_rows": int(audit["eval_leakage_rows"].sum()),
    "fold_audit": audit.to_dict(orient="records"),
    "conclusion": [
        "The selected surrogate recipe clears the wf2020 diagnostic return/MaxDD gates on a preview path, but this is not an official replay or promotion result.",
        "The durable output is the three-fold recovery-participation target contract plus BUY/SELL feature-state comparison sidecars for the next checkpoint replay/dashboard readout.",
    ],
})

write_json(SIDE_CARS["selection"], selection)
write_json(SIDE_CARS["summary"], summary)

conclusion = f"""# {EXPERIMENT} conclusion

Decision: `{summary['decision']}`.

Selected recipe: `dd12_cap15_recovery65_bridge35_mom02` with risk-only cap `{SELECTED_RECIPE['risk_cap']:.2f}`, recovery floor `{SELECTED_RECIPE['recovery_floor']:.2f}`, and conflict bridge `{SELECTED_RECIPE['conflict_bridge_floor']:.2f}`.

Preview metrics are diagnostic only: return `{selected_preview_metrics['return_pct']:.2f}%`, MaxDD `{selected_preview_metrics['max_dd_pct']:.2f}%`, surrogate gate `{selected_preview_metrics['surrogate_gate']}`. Base near-miss was return `{base_near_metrics['return_pct']:.2f}%`, MaxDD `{base_near_metrics['max_dd_pct']:.2f}%`.

The non-leaking target contract is `{rel(SIDE_CARS['target_contract'])}` with `{len(contract)}` rows and `{int((contract['target_delta_vs_source'].abs() > 1e-9).sum())}` modified rows. Eval leakage rows: `{int(audit['eval_leakage_rows'].sum())}`.

Next action: rerun three-fold checkpoint replay with this target contract. Promotion and actor retention remain blocked until official replay passes `wf_2020`, `wf_2022`, and `wf_2023_2024`.
"""
SIDE_CARS["conclusion"].write_text(conclusion, encoding="utf-8")

print(json.dumps(summary, indent=2, ensure_ascii=False)[:4000])

{
  "experiment": "wf2020_recovery_participation_target_v1",
  "created_at": "2026-06-14T03:27:37.429395+00:00",
  "decision": "wf2020_recovery_participation_target_ready_for_checkpoint_replay_no_promotion",
  "selected_recipe": {
    "strategy_dd_threshold": 0.12,
    "risk_cap": 0.15,
    "recovery_floor": 0.65,
    "conflict_bridge_floor": 0.35,
    "recovery_mom5_threshold": 0.02,
    "scale_clip": 3.0
  },
  "selected_recipe_id": "dd12_cap15_recovery65_bridge35_mom02",
  "direct_comparison_baseline": [
    "cap10_three_fold_checkpoint_metric_replay_v2:ppo_retained_iter_4",
    "wf2020_checkpoint_near_miss_overlay_screen_v1",
    "fold_local_bull_reentry_participation_target_v1"
  ],
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "reference_baseline": "soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05",
  "closest_historical_success_case": "cap10_three_fold_checkpoint_metric_replay_v2 passed wf_2022 and wf_2023_2024 but failed wf_2020.",
  "closest_historica

In [8]:
display(Markdown(
    f"""
## Conclusion

Decision: `{summary['decision']}`.

The selected recipe writes a non-leaking three-fold target contract at `{rel(SIDE_CARS['target_contract'])}` and separate 2020 preview evidence at `{rel(SIDE_CARS['selected_preview_daily'])}`. The preview is diagnostic only: return `{selected_preview_metrics['return_pct']:.2f}%`, MaxDD `{selected_preview_metrics['max_dd_pct']:.2f}%`, surrogate gate `{selected_preview_metrics['surrogate_gate']}`.

BUY/SELL feature comparison is written to `{rel(SIDE_CARS['buy_sell_feature_comparison'])}` so the dashboard can show the action-state differences instead of treating every model as if it used the same features.

Promotion and actor retention remain blocked. Next action: `rerun_three_fold_checkpoint_replay_with_recovery_participation_target_contract`.
""".strip()
))

## Conclusion

Decision: `wf2020_recovery_participation_target_ready_for_checkpoint_replay_no_promotion`.

The selected recipe writes a non-leaking three-fold target contract at `runtime/model_selection/card/wf2020_recovery_participation_target_v1/recovery_participation_three_fold_target_contract.csv` and separate 2020 preview evidence at `runtime/model_selection/card/wf2020_recovery_participation_target_v1/selected_recovery_preview_daily.csv`. The preview is diagnostic only: return `73.07%`, MaxDD `15.35%`, surrogate gate `True`.

BUY/SELL feature comparison is written to `runtime/model_selection/card/wf2020_recovery_participation_target_v1/buy_sell_feature_comparison.csv` so the dashboard can show the action-state differences instead of treating every model as if it used the same features.

Promotion and actor retention remain blocked. Next action: `rerun_three_fold_checkpoint_replay_with_recovery_participation_target_contract`.